In [110]:
import pandas as pd
import requests
from io import StringIO
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import re
import numpy as np
import plotly.express as px

## ***Coletando os dados de número de jogadores***

### Coleta dos dados do League of Leagends

In [111]:
headers = {
    'User-Agent': 'Mozilla/5.0'
}

url_lol = 'https://turbosmurfs.gg/article/league-of-legends-player-count-and-statistics'

response = requests.get(url_lol, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

texto = soup.get_text(' ', strip=True)

padrao = (
    r'(20\d{2}|2011|2012|2013|2014|2015|2016|2017|2018|2019)'
    r'\s+(\d+)'
    r'\s+Million'
    r'\s*([+-]?\d+(?:\.\d+)?%)'
)

dados = re.findall(padrao, texto)

df_lol = pd.DataFrame(
    dados,
    columns=[
        'Ano',
        'Jogadores_Ativos_Mensais',
        'Crescimento_Percentual'
    ]
)

df_lol['Ano'] = df_lol['Ano'].astype(int)

df_lol['Jogadores_Ativos_Mensais'] = (
    df_lol['Jogadores_Ativos_Mensais']
    .astype(float)
)

df_lol['Crescimento_Percentual'] = (
    df_lol['Crescimento_Percentual']
    .str.replace('%', '', regex=False)
    .astype(float)
)

df_lol.to_csv(
    'jogadores_lol.csv',
    index=False,
    encoding='utf-8-sig'
)

##### ***Obs.: a Riot Games não distribui publicamente os dados de seus jogos, então terei que trabalhar com estimativas do número de jogadores.***

### Função para coletar os dados do site Steamplayercount

In [112]:
def coletar_steam(url, nome_csv):

    response = requests.get(url, headers=headers)

    html = StringIO(response.text)

    tabelas = pd.read_html(html)

    df = tabelas[1].copy()

    df = df.rename(columns={
        'Month': 'Mês',
        'Peak': 'Pico',
        'Gain': 'Ganho',
        '% Gain': 'Ganho%',
        'Min Daily Peak': 'Pico_Mínimo_Diário',
        'Avg Daily Peak': 'Pico_Médio_Diário'
    })

    df.to_csv(
        nome_csv,
        index=False,
        encoding='utf-8-sig'
    )

    return df

### Coleta dos dados

In [113]:
print('Coletando Dota 2...')

df_dota2 = coletar_steam(
    'https://steamplayercount.com/app/570',
    'jogadores_dota2.csv'
)

print('Dota 2 concluído!')


print('Coletando SMITE...')

df_smite = coletar_steam(
    'https://steamplayercount.com/app/386360',
    'jogadores_smite.csv'
)

print('SMITE concluído!')

Coletando Dota 2...
Dota 2 concluído!
Coletando SMITE...
SMITE concluído!


## ***Leitura e Tratamento dos arquivos CSV***

In [114]:
df_lol = pd.read_csv('jogadores_lol.csv')
df_dota2 = pd.read_csv('jogadores_dota2.csv')
df_smite = pd.read_csv('jogadores_smite.csv')

### League of Legends

In [115]:
df_lol['Ano'] = df_lol['Ano'].astype(int)

df_lol['Jogadores_Ativos_Mensais'] = (
    df_lol['Jogadores_Ativos_Mensais']
    .astype(float)
)

df_lol['Crescimento_Percentual'] = (
    df_lol['Crescimento_Percentual']
    .astype(float)
)

# Salva as alterações
df_lol.to_csv(
    'jogadores_lol.csv',
    index=False,
    encoding='utf-8-sig'
)

In [116]:
df_lol.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Ano                       15 non-null     int64  
 1   Jogadores_Ativos_Mensais  15 non-null     float64
 2   Crescimento_Percentual    15 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 492.0 bytes


In [117]:
df_lol.head(25)

,Ano,Jogadores_Ativos_Mensais,Crescimento_Percentual
0,2011,15.0,200.0
1,2012,32.0,113.0
2,2013,67.0,110.0
3,2014,70.0,4.5
4,2015,90.0,28.0
5,2016,100.0,11.0
6,2017,81.0,-18.9
7,2018,75.0,-7.5
8,2019,116.0,54.5
9,2020,137.0,18.0


### Dota 2

In [118]:
df_dota2['Mês'] = pd.to_datetime(df_dota2['Mês'])

df_dota2['Ano'] = df_dota2['Mês'].dt.year

df_dota2_anual = (
    df_dota2
    .groupby('Ano', as_index=False)['Pico']
    .mean()
)

df_dota2_anual['Crescimento_Percentual'] = (
    df_dota2_anual['Pico']
    .pct_change()
    .mul(100)
    .round(0)
    .fillna(0)
)

df_dota2_anual.to_csv(
    'jogadores_dota2_anual.csv',
    index=False,
    encoding='utf-8-sig'
)

C:\Users\drrec\AppData\Local\Temp\ipykernel_12788\1038561284.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_dota2['Mês'] = pd.to_datetime(df_dota2['Mês'])


In [119]:
df_dota2.info()

<class 'pandas.DataFrame'>
RangeIndex: 174 entries, 0 to 173
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Mês                 174 non-null    datetime64[us]
 1   Pico                174 non-null    int64         
 2   Ganho               174 non-null    str           
 3   Ganho%              174 non-null    str           
 4   Pico_Mínimo_Diário  174 non-null    int64         
 5   Pico_Médio_Diário   174 non-null    int64         
 6   Ano                 174 non-null    int32         
dtypes: datetime64[us](1), int32(1), int64(3), str(2)
memory usage: 9.0 KB


In [120]:
df_dota2.head()

,Mês,Pico,Ganho,Ganho%,Pico_Mínimo_Diário,Pico_Médio_Diário,Ano
0,2026-02-01,870671,+13161,+2%,743693,788656,2026
1,2026-01-01,857510,-47739,-5%,705157,771469,2026
2,2025-12-01,905249,-3840,-1%,687249,800206,2025
3,2025-11-01,909089,+45253,+5%,699238,798545,2025
4,2025-10-01,863836,-83252,-9%,716015,767147,2025


### Smite

In [121]:
df_smite['Mês'] = pd.to_datetime(df_smite['Mês'])

df_smite['Ano'] = df_smite['Mês'].dt.year

df_smite_anual = (
    df_smite
    .groupby('Ano', as_index=False)['Pico']
    .mean()
)

df_smite_anual['Crescimento_Percentual'] = (
    df_smite_anual['Pico']
    .pct_change()
    .mul(100)
    .round(0)
    .fillna(0)
)

df_smite_anual.to_csv(
    'jogadores_smite_anual.csv',
    index=False,
    encoding='utf-8-sig'
)

C:\Users\drrec\AppData\Local\Temp\ipykernel_12788\1586720351.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_smite['Mês'] = pd.to_datetime(df_smite['Mês'])


In [122]:
df_smite.info()

<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Mês                 126 non-null    datetime64[us]
 1   Pico                126 non-null    int64         
 2   Ganho               126 non-null    str           
 3   Ganho%              126 non-null    str           
 4   Pico_Mínimo_Diário  126 non-null    int64         
 5   Pico_Médio_Diário   126 non-null    int64         
 6   Ano                 126 non-null    int32         
dtypes: datetime64[us](1), int32(1), int64(3), str(2)
memory usage: 6.5 KB


In [123]:
df_smite.head()

,Mês,Pico,Ganho,Ganho%,Pico_Mínimo_Diário,Pico_Médio_Diário,Ano
0,2026-02-01,2238,-351,-14%,1703,1884,2026
1,2026-01-01,2589,+287,+13%,1912,2148,2026
2,2025-12-01,2302,-4,0%,1703,1943,2025
3,2025-11-01,2306,+44,+2%,1763,1948,2025
4,2025-10-01,2262,-36,-2%,1655,1886,2025


## ***Padronização dos DataFrames***

In [124]:
#League of Legends

df_lol_anual = df_lol.rename(
    columns={
        'Jogadores_Ativos_Mensais': 'Valor'
    }
)

df_lol_anual['Jogo'] = 'League of Legends'

df_lol_anual = df_lol_anual[
    ['Ano', 'Valor', 'Jogo']
]

#Dota 2

df_dota2_anual = df_dota2_anual.rename(
    columns={
        'Pico': 'Valor'
    }
)

df_dota2_anual['Jogo'] = 'Dota 2'

df_dota2_anual = df_dota2_anual[
    ['Ano', 'Valor', 'Jogo']
]

#SMITE

df_smite_anual = df_smite_anual.rename(
    columns={
        'Pico': 'Valor'
    }
)

df_smite_anual['Jogo'] = 'SMITE'

df_smite_anual = df_smite_anual[
    ['Ano', 'Valor', 'Jogo']
]

### Unindo so DataFrames

In [125]:
df_todos = pd.concat(
    [
        df_lol_anual,
        df_dota2_anual,
        df_smite_anual
    ],
    ignore_index=True
)

df_todos = df_todos.sort_values(
    ['Jogo', 'Ano']
)

df_todos.to_csv(
    'comparacao_mobas.csv',
    index=False,
    encoding='utf-8-sig'
)

### Normalizar pelo maior pico de cada jogo

In [126]:
df_todos = df_todos.sort_values(['Jogo', 'Ano'])

df_todos['Indice_Pico'] = (
    df_todos
    .groupby('Jogo')['Valor']
    .transform(
        lambda x: (x / x.max()) * 100
        )
)

df_todos.to_csv(
    'comparacao_base100.csv',
    index=False,
    encoding='utf-8-sig'
)

#Com filtro

ano_base = 2015
ano_limite = 2025

df_todos_filtro = df_todos[
    df_todos['Ano'].between(
        ano_base,
        ano_limite
    )
]

df_todos_filtro = df_todos_filtro.sort_values(
    ['Jogo', 'Ano']
)

df_todos_filtro['Base_100'] = (
    df_todos_filtro
    .groupby('Jogo')['Valor']
    .transform(
        lambda x: (x / x.max()) * 100
    )
)
df_todos_filtro.to_csv(
    'comparacao_base100_filtro.csv',
    index=False,
    encoding='utf-8-sig'
)

In [127]:
df_todos_filtro.info()

<class 'pandas.DataFrame'>
Index: 33 entries, 19 to 41
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Ano          33 non-null     int64  
 1   Valor        33 non-null     float64
 2   Jogo         33 non-null     str    
 3   Indice_Pico  33 non-null     float64
 4   Base_100     33 non-null     float64
dtypes: float64(3), int64(1), str(1)
memory usage: 1.5 KB


In [128]:
df_todos.head(33)

,Ano,Valor,Jogo,Indice_Pico
15,2011,8.351750e+03,Dota 2,0.749595
16,2012,9.393608e+04,Dota 2,8.431045
17,2013,4.448668e+05,Dota 2,39.928132
18,2014,8.209276e+05,Dota 2,73.680711
19,2015,9.852034e+05,Dota 2,88.424960
20,2016,1.114169e+06,Dota 2,100.000000
21,2017,9.121416e+05,Dota 2,81.867441
22,2018,7.856926e+05,Dota 2,70.518264
23,2019,8.595110e+05,Dota 2,77.143689
24,2020,7.137580e+05,Dota 2,64.061920


In [129]:
df_todos_filtro.head(33)

,Ano,Valor,Jogo,Indice_Pico,Base_100
19,2015,9.852034e+05,Dota 2,88.424960,88.424960
20,2016,1.114169e+06,Dota 2,100.000000,100.000000
21,2017,9.121416e+05,Dota 2,81.867441,81.867441
22,2018,7.856926e+05,Dota 2,70.518264,70.518264
23,2019,8.595110e+05,Dota 2,77.143689,77.143689
24,2020,7.137580e+05,Dota 2,64.061920,64.061920
25,2021,7.038941e+05,Dota 2,63.176604,63.176604
26,2022,7.930892e+05,Dota 2,71.182130,71.182130
27,2023,7.573639e+05,Dota 2,67.975682,67.975682
28,2024,7.876488e+05,Dota 2,70.693836,70.693836


## ***Coletando os dados da popularidade***

In [130]:
df_trends_moba =pd.read_csv("google_trends_mobas.csv")

In [131]:
df_trends_moba.head()

,Time,League of Legends,Dota 2,Smite
0,2004-01-01,3,0,0
1,2004-02-01,3,0,0
2,2004-03-01,4,0,0
3,2004-04-01,3,1,0
4,2004-05-01,4,1,0


In [132]:
df_trends_moba.info()

<class 'pandas.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Time               270 non-null    str  
 1   League of Legends  270 non-null    int64
 2   Dota 2             270 non-null    int64
 3   Smite              270 non-null    int64
dtypes: int64(3), str(1)
memory usage: 8.6 KB


In [133]:
df_trends_moba['Time'] = pd.to_datetime(df_trends_moba['Time'])

df_trends_moba['Ano'] = df_trends_moba['Time'].dt.year

df_trends_moba_anual = (
    df_trends_moba
    .groupby('Ano', as_index=False)
    .mean(numeric_only=True)
)
df_trends_moba_anual = df_trends_moba_anual.rename(
    columns={
        'Smite': 'SMITE'
    }
)

In [134]:
df_trends_moba_anual.head()

,Ano,League of Legends,Dota 2,SMITE
0,2004,3.833333,1.416667,0.0
1,2005,4.583333,4.833333,0.0
2,2006,5.250000,9.333333,0.0
3,2007,5.833333,12.333333,0.0
4,2008,5.750000,16.333333,0.0


In [135]:
df_trends_moba_anual_filtro = df_trends_moba_anual[
    df_trends_moba_anual['Ano'].between(
        ano_base,
        ano_limite
        )
]
df_trends_moba_anual_filtro.head(21)

,Ano,League of Legends,Dota 2,SMITE
11,2015,84.833333,30.833333,6.166667
12,2016,65.500000,22.250000,5.416667
13,2017,55.416667,15.833333,3.333333
14,2018,39.500000,10.750000,2.500000
15,2019,36.750000,9.166667,2.000000
16,2020,39.000000,7.333333,2.416667
17,2021,30.500000,6.916667,2.166667
18,2022,33.916667,9.000000,1.833333
19,2023,36.750000,7.083333,1.583333
20,2024,36.000000,9.083333,1.416667


In [136]:
#Sem o filtro de ano
df_trends_moba_anual.to_csv(
    'google_trends_mobas_editado.csv',
    index=False,
    encoding='utf-8-sig'
)

#Com o filtro de ano
df_trends_moba_anual_filtro.to_csv(
    'google_trends_mobas_editado_filtro.csv',
    index=False,
    encoding='utf-8-sig'
)

## ***Analizes***

In [137]:
df_todos_filtro = pd.read_csv('comparacao_base100_filtro.csv')

fig = px.line(
    df_todos_filtro,
    x='Ano',
    y='Base_100',
    color='Jogo',
    markers=True,
    color_discrete_map={
        'League of Legends': '#DF1717',
        'Dota 2': '#22C265',
        'SMITE': "#29B6E0"
    }
)

fig.update_layout(
    title='Evolução relativa do pico de jogadores dos MOBAs (com filtro)',
    xaxis_title='Ano',
    yaxis_title='Índice Base 100',
    template='plotly_white',
    hovermode='x unified'
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig.show()

Este gráfico apresenta a evolução anual do número de jogadores normalizada pelo maior pico histórico registrado para cada jogo. Dessa forma, cada série possui valor máximo igual a 100, permitindo comparar o quanto cada título esteve próximo ou distante de seu melhor desempenho ao longo dos anos. O período inicia em 2015 devido à disponibilidade de dados do jogo Smite.

O gráfico evidencia que, a partir de 2019, observa-se uma recuperação do jogo League of Legends, que se manteve durante 4 anos, diferente do jogo Smite, que também teve um crescimento no número de jogadores para o ano de 2020, a recuperação observada coincide com o período da pandemia de COVID-19, durante o qual houve aumento do consumo de jogos eletrônicos em diversas plataformas. Enquanto ao jogo Dota 2, apresentou oscilações menores que tenderam à queda ao longo dos anos, mas se manteve relativamente estável a partir de 2019. Dessa forma, os resultados não fornecem evidências de que o League of Legends esteja em processo de decadência quando comparado aos demais MOBAs analisados. Pelo contrário, entre os jogos considerados, foi o que apresentou a evolução relativa mais favorável durante o período analisado.

Observa-se que a redução do número de jogadores do jogo Smite antecede o anúncio do Smite 2 (27 de março de 2024). Esse comportamento levanta a hipótese de que a queda na base de jogadores possa ter influenciado a decisão da desenvolvedora de lançar uma sequência. Entretanto, apenas os dados apresentados não são suficientes para estabelecer essa relação causal.

Lembrando que a Riot Games não disponibiliza os dados de seus jogos, e o que foi usado é uma estimativa retirada do site turbosmurfs.gg.

In [141]:
df_todos = pd.read_csv('comparacao_base100.csv')

fig = px.line(
    df_todos,
    x='Ano',
    y='Indice_Pico',
    color='Jogo',
    markers=True,
    color_discrete_map={
        'League of Legends': '#DF1717',
        'Dota 2': '#22C265',
        'SMITE': '#46C0E5'
    }
)

fig.update_layout(
    title='Evolução relativa do pico de jogadores dos MOBAs',
    xaxis_title='Ano',
    yaxis_title='Percentual do pico histórico (%)',
    template='plotly_white',
    hovermode='x unified'
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig.show()

Com a análise do gráfico sem o filtro de ano. Observa-se que o Dota 2 atingiu seu pico histórico em 2016 e, a partir desse momento, apresentou uma tendência de redução, estabilizando-se nos anos seguintes. O League of Legends, por sua vez, apresentou crescimento gradual até atingir seu pico histórico entre 2022 e 2023, mantendo-se posteriormente próximo desse patamar, mesmo com uma leve redução nos anos finais.

O SMITE também apresentou crescimento até alcançar seu maior pico em 2021. Entretanto, a partir desse período, observa-se uma queda contínua, uma redução bastante acentuada entre 2024 e 2025, quando o jogo passou a representar apenas uma pequena parcela de seu pico histórico.

É possível observar que, entre 2016 e 2018, os três jogos apresentaram uma redução em relação aos seus respectivos picos históricos, embora em momentos e intensidades diferentes. Em 2019, League of Legends e Dota 2 apresentavam níveis relativos semelhantes em relação aos seus picos, mas suas trajetórias passaram a divergir nos anos seguintes. Enquanto o League of Legends continuou crescendo até alcançar seu maior pico histórico, o Dota 2 permaneceu relativamente estável, porém distante do seu melhor desempenho.

In [139]:
#Sem o filtro de ano
df_tdmobas_trends = pd.read_csv('google_trends_mobas_editado.csv')

df_trends_long = df_tdmobas_trends.melt(
    id_vars='Ano',
    var_name='Jogo',
    value_name='Interesse'
)

fig = px.bar(
    df_trends_long,
    x='Ano',
    y='Interesse',
    color='Jogo',
    barmode='group',
    color_discrete_map={
        'League of Legends': "#E3C830",
        'Dota 2': "#E330A9",
        'SMITE': "#30DEE3"
    }
)

fig.update_layout(
    title='Comparação do interesse de pesquisa sem o filtro de ano (Google Trends)',
    xaxis_title='Ano',
    yaxis_title='Índice de interesse',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

In [140]:
#Com o filtro de ano
df_tdmobas_trends_filtro = pd.read_csv('google_trends_mobas_editado_filtro.csv')

df_trends_long_filtro = df_tdmobas_trends_filtro.melt(
    id_vars='Ano',
    var_name='Jogo',
    value_name='Interesse'
)

fig = px.bar(
    df_trends_long_filtro,
    x='Ano',
    y='Interesse',
    color='Jogo',
    barmode='group',
    color_discrete_map={
        'League of Legends': "#E3C830",
        'Dota 2': "#E330A9",
        'SMITE': "#30DEE3"
    }
)

fig.update_layout(
    title='Comparação do interesse de pesquisa com o filtro de ano (Google Trends)',
    xaxis_title='Ano',
    yaxis_title='Índice de interesse',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()